In [1]:
!pip install tensorboardX rdkit
!pip install rdkit-pypi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 2.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 48.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 55.2 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as Data
#torch.manual_seed(8) # for reproduce


from sklearn.model_selection import LeaveOneOut
import time
from tqdm import tqdm
import numpy as np
import gc
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/Posdoc/Native_AFP/code/')
sys.setrecursionlimit(50000)
import pickle
torch.backends.cudnn.benchmark = True
torch.set_default_tensor_type('torch.cuda.FloatTensor')
# from tensorboardX import SummaryWriter
torch.nn.Module.dump_patches = True
import copy
import pandas as pd
#then import my own modules
from AttentiveFP import Fingerprint_viz, save_smiles_dicts, get_smiles_dicts, get_smiles_array, moltosvg_highlight
from rdkit import Chem
# from rdkit.Chem import AllChem
from rdkit.Chem import QED
from rdkit.Chem import rdMolDescriptors, MolSurf
from rdkit.Chem.Draw import SimilarityMaps
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
%matplotlib inline
from numpy.polynomial.polynomial import polyfit
import matplotlib.pyplot as plt
from matplotlib import gridspec
import matplotlib.cm as cm
import matplotlib
import seaborn as sns; sns.set_style("darkgrid")
from IPython.display import SVG, display
import sascorer
import itertools
from sklearn.metrics import r2_score
import scipy
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
import pickle
import os
import time
from rdkit import Chem
from sklearn.model_selection import LeaveOneOut
from collections import defaultdict


device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

/usr/local/lib/python3.11/dist-packages/torch/__init__.py:614: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:451.)
  _C._set_default_tensor_type(t)


cuda


In [3]:

class Fingerprint(nn.Module):
    def __init__(self, radius, T, input_feature_dim, input_bond_dim,
                 fingerprint_dim, output_units_num, p_dropout):
        super(Fingerprint, self).__init__()
        
        self.radius = radius
        self.T = T
        
        # Atom embedding
        self.atom_fc = nn.Linear(input_feature_dim, fingerprint_dim)
        self.neighbor_fc = nn.Linear(input_feature_dim + input_bond_dim, fingerprint_dim)
        self.gru_cells = nn.ModuleList([nn.GRUCell(fingerprint_dim, fingerprint_dim) for _ in range(radius)])
        self.align_layers = nn.ModuleList([nn.Linear(2 * fingerprint_dim, 1) for _ in range(radius)])
        self.attend_layers = nn.ModuleList([nn.Linear(fingerprint_dim, fingerprint_dim) for _ in range(radius)])
        
        # Molecule embedding
        self.mol_gru_cell = nn.GRUCell(fingerprint_dim, fingerprint_dim)
        self.mol_align = nn.Linear(2 * fingerprint_dim, 1)
        self.mol_attend = nn.Linear(fingerprint_dim, fingerprint_dim)
        
        self.dropout = nn.Dropout(p=p_dropout)
        self.output = nn.Linear(fingerprint_dim, output_units_num)
        self.pairwise_output = nn.Linear(fingerprint_dim * 2, output_units_num)
        
    def forward(self, atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1,
                atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2):
        # Process first molecule
        atom_feature1, mol_feature1 = self.process_single_molecule(
            atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1
        )

        # Process second molecule
        atom_feature2, mol_feature2 = self.process_single_molecule(
            atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2
        )

        # Concatenate molecular features
        combined_feature = torch.cat([mol_feature1, mol_feature2], dim=1)

        # Final pairwise prediction
        pairwise_prediction = self.pairwise_output(self.dropout(combined_feature))

        return atom_feature1, atom_feature2, pairwise_prediction

    def process_single_molecule(self, atom_list, bond_list, atom_degree_list, bond_degree_list, atom_mask):
        batch_size, mol_length, num_atom_feat = atom_list.size()
        atom_mask = atom_mask.unsqueeze(2)

        atom_feature = F.leaky_relu(self.atom_fc(atom_list))
        neighbor_feature = self.get_neighbor_feature(atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size)

        attend_mask, softmax_mask = self.create_masks(atom_degree_list, mol_length)

        atom_feature = self.apply_atom_attention(atom_feature, neighbor_feature, attend_mask, softmax_mask)
        mol_feature = torch.sum(F.relu(atom_feature) * atom_mask, dim=-2)

        mol_feature = self.apply_molecule_attention(atom_feature, mol_feature, atom_mask)

        return atom_feature, mol_feature


    def get_neighbor_feature(self, atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size):
        bond_neighbor = torch.stack([bond_list[i][bond_degree_list[i]] for i in range(batch_size)], dim=0)
        atom_neighbor = torch.stack([atom_list[i][atom_degree_list[i]] for i in range(batch_size)], dim=0)
        neighbor_feature = torch.cat([atom_neighbor, bond_neighbor], dim=-1)
        return F.leaky_relu(self.neighbor_fc(neighbor_feature))

    def create_masks(self, atom_degree_list, mol_length):
        attend_mask = (atom_degree_list != mol_length - 1).float().unsqueeze(-1)
        softmax_mask = torch.where(atom_degree_list == mol_length - 1, torch.tensor(-9e8).cuda(), torch.tensor(0.).cuda()).unsqueeze(-1)
        return attend_mask, softmax_mask

    def apply_atom_attention(self, atom_feature, neighbor_feature, attend_mask, softmax_mask):
        #print("Applying atom-level attention")
        batch_size, mol_length = atom_feature.shape[:2]
        
        for d in range(self.radius):
            atom_feature_expand = atom_feature.unsqueeze(-2).expand(-1, -1, neighbor_feature.size(2), -1)
            feature_align = torch.cat([atom_feature_expand, neighbor_feature], dim=-1)
            
            align_score = F.leaky_relu(self.align_layers[d](self.dropout(feature_align))) + softmax_mask
            attention_weight = F.softmax(align_score, -2) * attend_mask
            
            context = torch.sum(attention_weight * self.attend_layers[d](self.dropout(neighbor_feature)), -2)
            context = F.elu(context)
            
            atom_feature = self.gru_cells[d](
                context.view(batch_size * mol_length, -1),
                atom_feature.view(batch_size * mol_length, -1)
            ).view(batch_size, mol_length, -1)
            
            neighbor_feature = F.relu(atom_feature).unsqueeze(-2).expand(-1, -1, neighbor_feature.size(2), -1)
        
        return atom_feature

    def apply_molecule_attention(self, atom_feature, mol_feature, atom_mask):
        #print("Applying mol-level attention")
        batch_size, mol_length = atom_feature.shape[:2]
        mol_softmax_mask = torch.where(atom_mask.squeeze() == 0, torch.tensor(-9e8).cuda(), torch.tensor(0.).cuda())
        
        for _ in range(self.T):
            mol_prediction_expand = mol_feature.unsqueeze(-2).expand(-1, mol_length, -1)
            mol_align = torch.cat([mol_prediction_expand, atom_feature], dim=-1)
            mol_align_score = F.leaky_relu(self.mol_align(mol_align)) + mol_softmax_mask.unsqueeze(-1)
            mol_attention_weight = F.softmax(mol_align_score, -2) * atom_mask
            
            mol_context = torch.sum(mol_attention_weight * self.mol_attend(self.dropout(atom_feature)), -2)
            mol_context = F.elu(mol_context)
            mol_feature = self.mol_gru_cell(mol_context, mol_feature)
        
        return mol_feature

In [4]:
def custom_cv_split(df):
    all_smiles = set()
    for pair in df['SMILES_pair']:
        smiles1, smiles2 = pair.split(',')
        all_smiles.add(smiles1)
        all_smiles.add(smiles2)
    
    n_splits = len(all_smiles)
    
    for test_smile in all_smiles:
        test_indices = []
        train_indices = []
        for idx, pair in enumerate(df['SMILES_pair']):
            smiles1, smiles2 = pair.split(',')
            if test_smile in (smiles1, smiles2):
                test_indices.append(idx)
            else:
                train_indices.append(idx)
        yield train_indices, test_indices, test_smile

#_____________________________________________________________________________________________________________________________________________ 
targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']
def validate(model, val_df, feature_dicts, loss_function, device, batch_size, return_predictions=False, test_smile=None):
    model.eval()
    total_loss = 0
    all_predictions = []
    all_true_values = []
    result_dict = {}

    with torch.no_grad():
        for i in range(0, len(val_df), batch_size):
            batch_df = val_df.iloc[i:i+batch_size]
            
            smile_pairs = batch_df.SMILES_pair.values
            y_val = batch_df[targets].values
            
            smiles_list1, smiles_list2 = zip(*[pair.split(',') for pair in smile_pairs])
            
            x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array(smiles_list1, feature_dicts)
            x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array(smiles_list2, feature_dicts)
            
            x_atom1, x_bonds1 = torch.Tensor(x_atom1).to(device), torch.Tensor(x_bonds1).to(device)
            x_atom_index1, x_bond_index1 = torch.LongTensor(x_atom_index1).to(device), torch.LongTensor(x_bond_index1).to(device)
            x_mask1 = torch.Tensor(x_mask1).to(device)
            
            x_atom2, x_bonds2 = torch.Tensor(x_atom2).to(device), torch.Tensor(x_bonds2).to(device)
            x_atom_index2, x_bond_index2 = torch.LongTensor(x_atom_index2).to(device), torch.LongTensor(x_bond_index2).to(device)
            x_mask2 = torch.Tensor(x_mask2).to(device)
            
            output_tuple = model(x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1,
                                 x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2)
            
            predictions = output_tuple[2]
            
            loss = loss_function(predictions, torch.Tensor(y_val).to(device))
            total_loss += loss.item()
            
            all_predictions.extend(predictions.cpu().numpy())
            all_true_values.extend(y_val)

            # Populate result_dict
            for j, smile_pair in enumerate(smile_pairs):
                smile1, smile2 = smile_pair.split(',')
                if smile1 == test_smile or smile2 == test_smile:
                    if test_smile not in result_dict:
                        result_dict[test_smile] = {}
                    result_dict[test_smile][smile_pair] = {}
                    for k, target in enumerate(targets):
                        result_dict[test_smile][smile_pair][target] = {
                            "actual": y_val[j][k],
                            "predicted": predictions[j][k].item()
                        }
    
    avg_loss = total_loss / (len(val_df) // batch_size + 1)
    r2 = r2_score(all_true_values, all_predictions)
    
    if return_predictions:
        if test_smile:
            return avg_loss, r2, all_true_values, all_predictions, result_dict
        else:
            print('NO TEST SMILE!!!!!')
            return avg_loss, r2, all_true_values, all_predictions, result_dict
    else:
        return avg_loss, r2



#_____________________________________________________________________________________________________________________________________________        

targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']
def prepare_model_and_data_for_relative_learning(raw_filename, target_name='Calx', targets=None, random_seed=888, batch_size=50, epochs=800, p_dropout=0.1, fingerprint_dim=210, weight_decay=5, learning_rate=3, radius=4, T=2, per_target_output_units_num=1):
    if targets is None:
        targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

    feature_filename = raw_filename.replace('.csv', '.pickle')
    smiles_targets_df = pd.read_csv(raw_filename)

    smilesList = smiles_targets_df.SMILES.values
    print("number of all smiles: ", len(smilesList))

    atom_num_dist = []
    remained_smiles = []
    canonical_smiles_list = []
    for smiles in smilesList:
        try:
            mol = Chem.MolFromSmiles(smiles)
            atom_num_dist.append(len(mol.GetAtoms()))
            remained_smiles.append(smiles)
            canonical_smiles_list.append(Chem.MolToSmiles(Chem.MolFromSmiles(smiles), isomericSmiles=True))
        except:
            print(smiles)
            pass
    print("number of successfully processed smiles: ", len(remained_smiles))

    smiles_targets_df = smiles_targets_df[smiles_targets_df["SMILES"].isin(remained_smiles)]
    smiles_targets_df['cano_smiles'] = canonical_smiles_list

    if os.path.isfile(feature_filename):
        feature_dicts = pickle.load(open(feature_filename, "rb"))
    else:
        feature_dicts = save_smiles_dicts(smilesList, filename)
    
    feature_dicts = get_smiles_dicts(smilesList)

    remained_df = smiles_targets_df[smiles_targets_df["cano_smiles"].isin(feature_dicts['smiles_to_atom_mask'].keys())]

    # Create pairs of SMILES and calculate relative targets
    smile_pairs = []
    relative_targets = []
    
    for i in range(len(remained_df)):
        for j in range(len(remained_df)):
            if i != j:  # Exclude self-pairs
                smile1 = remained_df.iloc[i]['cano_smiles']
                smile2 = remained_df.iloc[j]['cano_smiles']
                target1 = remained_df.iloc[i][targets].values
                target2 = remained_df.iloc[j][targets].values
                
                # Calculate relative target (difference)
                rel_target = target1 - target2
                
                smile_pairs.append(f"{smile1},{smile2}")
                relative_targets.append(rel_target)

    # Create DataFrame
    df = pd.DataFrame(relative_targets, columns=targets)
    df['SMILES_pair'] = smile_pairs
    df = df[['SMILES_pair'] + targets]  # Reorder columns

    # Prepare input features for the model
    feature_data = []
    
    for smile_pair in smile_pairs:
        smile1, smile2 = smile_pair.split(',')
        x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array([smile1], feature_dicts)
        x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array([smile2], feature_dicts)
        
        feature_data.append({
            'x_atom1': x_atom1,
            'x_bonds1': x_bonds1,
            'x_atom_index1': x_atom_index1,
            'x_bond_index1': x_bond_index1,
            'x_mask1': x_mask1,
            'x_atom2': x_atom2,
            'x_bonds2': x_bonds2,
            'x_atom_index2': x_atom_index2,
            'x_bond_index2': x_bond_index2,
            'x_mask2': x_mask2
        })

    num_atom_features = feature_data[0]['x_atom1'].shape[-1]
    num_bond_features = feature_data[0]['x_bonds1'].shape[-1]

    loss_function = nn.MSELoss()
    model = Fingerprint(radius, T, num_atom_features, num_bond_features, fingerprint_dim, len(targets), p_dropout)
    model.to(device)

    optimizer = optim.Adam(model.parameters(), 10**-learning_rate, weight_decay=10**-weight_decay)

    return model, optimizer, loss_function, df, feature_data, feature_filename,remained_df

# Example usage:
model, optimizer, loss_function, df, feature_data, feature_filename ,remained_df= prepare_model_and_data_for_relative_learning('/notebooks/data/cal_abs.csv')
                                                          

number of all smiles:  40
number of successfully processed smiles:  40


In [5]:
if os.path.isfile(feature_filename):
    feature_dicts = pickle.load(open(feature_filename, "rb" ))
else:
    feature_dicts = save_smiles_dicts(smilesList,filename)
# feature_dicts = get_smiles_dicts(smilesList)


for index, row in df.iterrows():
    # Split the SMILES pair
    smile1, smile2 = row['SMILES_pair'].split(',')
    
    # Get features for smile1
    x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array([smile1], feature_dicts)

    # Get features for smile2
    x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array([smile2], feature_dicts)
print(x_atom2.shape)

(1, 73, 39)


In [6]:
def train_and_evaluate(model, df, feature_dicts, optimizer, loss_function, num_epochs=100, batch_size=32, patience=10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    fold_results = []
    all_true_values = []
    all_predictions = []
    final_dic= {}
    initial_state = {k: v.clone() for k, v in model.state_dict().items()}
    
    n_splits = len(set(smiles for pair in df['SMILES_pair'] for smiles in pair.split(',')))
    
    for fold, (train_val_index, test_index, test_smile) in enumerate(custom_cv_split(df), 1):
        print(f"\nFold {fold}/{n_splits}")
        print(f"Testing SMILES: {test_smile}")
        
        # Reset the model for each fold
        model.load_state_dict(initial_state)
        
        # Reset optimizer state
        optimizer.state = defaultdict(dict)
        
        # Split the data into train+val and test
        train_val_df = df.iloc[train_val_index]
        test_df = df.iloc[test_index]
        
        # Check for test SMILES in train_val set
        train_val_smiles = set()
        for pair in train_val_df['SMILES_pair']:
            smiles1, smiles2 = pair.split(',')
            train_val_smiles.add(smiles1)
            train_val_smiles.add(smiles2)
        
        if test_smile in train_val_smiles:
            print(f"WARNING: Test SMILES {test_smile} found in train_val set!")
        else:
            print(f"Verification: Test SMILES {test_smile} not found in train_val set.")
        
        # Further split train_val into train and validation
        train_df, val_df = train_test_split(train_val_df, test_size=0.2, random_state=42)
        
        # Additional check for test SMILES in train and val sets
        train_smiles = set()
        for pair in train_df['SMILES_pair']:
            smiles1, smiles2 = pair.split(',')
            train_smiles.add(smiles1)
            train_smiles.add(smiles2)
        
        val_smiles = set()
        for pair in val_df['SMILES_pair']:
            smiles1, smiles2 = pair.split(',')
            val_smiles.add(smiles1)
            val_smiles.add(smiles2)
        
        if test_smile in train_smiles:
            print(f"WARNING: Test SMILES {test_smile} found in training set!")
        else:
            print(f"Verification: Test SMILES {test_smile} not found in training set.")
        
        if test_smile in val_smiles:
            print(f"WARNING: Test SMILES {test_smile} found in validation set!")
        else:
            print(f"Verification: Test SMILES {test_smile} not found in validation set.")
        
        best_val_loss = float('inf')
        epochs_no_improve = 0
        
        for epoch in range(num_epochs):
            # Training code (unchanged)
        

            model.train()
            total_loss = 0
            
            train_df = train_df.sample(frac=1).reset_index(drop=True)
            
            for i in range(0, len(train_df), batch_size):
                batch_df = train_df.iloc[i:i+batch_size]
                
                smile_pairs = batch_df.SMILES_pair.values
                y_val = batch_df[targets].values
                
                smiles_list1, smiles_list2 = zip(*[pair.split(',') for pair in smile_pairs])
                
                x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array(smiles_list1, feature_dicts)
                x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array(smiles_list2, feature_dicts)
                
                x_atom1, x_bonds1 = torch.Tensor(x_atom1).to(device), torch.Tensor(x_bonds1).to(device)
                x_atom_index1, x_bond_index1 = torch.LongTensor(x_atom_index1).to(device), torch.LongTensor(x_bond_index1).to(device)
                x_mask1 = torch.Tensor(x_mask1).to(device)
                
                x_atom2, x_bonds2 = torch.Tensor(x_atom2).to(device), torch.Tensor(x_bonds2).to(device)
                x_atom_index2, x_bond_index2 = torch.LongTensor(x_atom_index2).to(device), torch.LongTensor(x_bond_index2).to(device)
                x_mask2 = torch.Tensor(x_mask2).to(device)
                
                output_tuple = model(x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1,
                                     x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2)
                
                predictions = output_tuple[2]
                
                optimizer.zero_grad()
                loss = loss_function(predictions, torch.Tensor(y_val).to(device))
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            avg_loss = total_loss / (len(train_df) // batch_size + 1)
            print(f'Epoch {epoch+1}/{num_epochs} done. Average Loss: {avg_loss}')
            
            # Validation step
            val_loss, val_r2 = validate(model, val_df, feature_dicts, loss_function, device, batch_size)
            print(f'Validation Loss: {val_loss}, Validation R2: {val_r2}')
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_no_improve = 0
                print(f'New best model found for fold {fold} with validation loss: {best_val_loss}')
                # Save the best model params as a pth file
                torch.save(model.state_dict(), f'/notebooks/code/saved_relative/best_model_fold_{fold}.pth')
            else:
                epochs_no_improve += 1
                
            if epochs_no_improve == patience:
                print(f'Early stopping triggered after {epoch+1} epochs')
                break
        

        # Evaluate on the test set for this fold
        model.load_state_dict(torch.load(f'/notebooks/code/saved_relative/best_model_fold_{fold}.pth'))
        
        test_loss, test_r2, true_values, predictions, result_dict = validate(model, test_df, feature_dicts, loss_function, device, batch_size, return_predictions=True,test_smile=test_smile)
        print(f'Test Loss for fold {fold}: {test_loss}, Test R2: {test_r2}')
        final_dic.update(result_dict)
        
        all_true_values.extend(true_values)
        all_predictions.extend(predictions)
        
        fold_results.append((best_val_loss, test_loss, test_r2))
        print(f'Fold {fold} completed. Best validation loss: {best_val_loss}, Test loss: {test_loss}, Test R2: {test_r2}')
    
    print('\nCross-validation completed.')
    val_losses, test_losses, test_r2s = zip(*fold_results)
    print(f'Average validation loss across folds: {np.mean(val_losses)}')
    print(f'Average test loss across folds: {np.mean(test_losses)}')
    print(f'Average test R2 across folds: {np.mean(test_r2s)}')
    
    overall_r2 = r2_score(all_true_values, all_predictions)
    print(f'Overall R2: {overall_r2}')
    
    return fold_results, overall_r2, final_dic


fold_results, overall_r2,final_dic = train_and_evaluate(model, df, feature_dicts, optimizer, loss_function, num_epochs=20, batch_size=32, patience=10)


Fold 1/40
Testing SMILES: O=C(Nc1cc2c(O)c(c1)Cc1cc(S(=O)(=O)O)cc(c1O)Cc1cc(S(=O)(=O)O)cc(c1O)Cc1cc(S(=O)(=O)O)cc(c1O)C2)OCc1ccccc1
Verification: Test SMILES O=C(Nc1cc2c(O)c(c1)Cc1cc(S(=O)(=O)O)cc(c1O)Cc1cc(S(=O)(=O)O)cc(c1O)Cc1cc(S(=O)(=O)O)cc(c1O)C2)OCc1ccccc1 not found in train_val set.
Verification: Test SMILES O=C(Nc1cc2c(O)c(c1)Cc1cc(S(=O)(=O)O)cc(c1O)Cc1cc(S(=O)(=O)O)cc(c1O)Cc1cc(S(=O)(=O)O)cc(c1O)C2)OCc1ccccc1 not found in training set.
Verification: Test SMILES O=C(Nc1cc2c(O)c(c1)Cc1cc(S(=O)(=O)O)cc(c1O)Cc1cc(S(=O)(=O)O)cc(c1O)Cc1cc(S(=O)(=O)O)cc(c1O)C2)OCc1ccccc1 not found in validation set.
Epoch 1/20 done. Average Loss: 6.3633679841694075
Validation Loss: 5.235592794418335, Validation R2: 0.0854637364738224
New best model found for fold 1 with validation loss: 5.235592794418335
Epoch 2/20 done. Average Loss: 5.220639009224741
Validation Loss: 4.922901153564453, Validation R2: 0.14277854852230953
New best model found for fold 1 with validation loss: 4.922901153564453
Epoch 3

In [7]:
with open('Relative_FP.pkl', 'wb') as file: 
      
    # A new file will be created 
    pickle.dump(final_dic, file) 